In [1]:
# ============================================================
# Step 6 setup
#
# Upload:
#   1. config.py
#
# Read from Google Drive:
#   2. Step 5 hidden-state zip
#
# This cell prepares:
#   /content/pilot_4/config.py
#   cfg.STEP6_INPUT_DIR
#   cfg.STEP6_OUTPUT_DIR
# ============================================================

import os
import sys
import shutil
import zipfile
import importlib
from pathlib import Path

from google.colab import files
from google.colab import drive

drive.mount("/content/drive")

# ------------------------------------------------------------
# 1. Create project structure
# ------------------------------------------------------------

PILOT_ROOT = "/content/pilot_4"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")
DATA_DIR = os.path.join(PILOT_ROOT, "data")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# ------------------------------------------------------------
# 2. Upload config.py
# ------------------------------------------------------------

print("Upload config.py")
uploaded_config = files.upload()

config_upload_name = None

for name in uploaded_config.keys():
    if name.startswith("config") and name.endswith(".py"):
        config_upload_name = name
        break

if config_upload_name is None:
    raise FileNotFoundError(
        f"No config*.py file was uploaded. Uploaded files: {list(uploaded_config.keys())}"
    )

config_target_path = os.path.join(PILOT_ROOT, "config.py")

shutil.copy(
    os.path.join("/content", config_upload_name),
    config_target_path,
)

# ------------------------------------------------------------
# 3. Load config
# ------------------------------------------------------------

import pilot_4.config as cfg
importlib.reload(cfg)

print("\nLoaded config:")
print("PILOT_ROOT:", cfg.PILOT_ROOT)
print("DATA_DIR:", cfg.DATA_DIR)
print("STEP6_INPUT_DIR:", cfg.STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", cfg.STEP6_OUTPUT_DIR)
print("STEP6_EXPERIMENT_NAME:", cfg.STEP6_EXPERIMENT_NAME)

# ------------------------------------------------------------
# 4. Prepare Step 6 input/output directories
# ------------------------------------------------------------

os.makedirs(cfg.STEP6_INPUT_DIR, exist_ok=True)

# Clear old Step 6 input files.
for old_path in Path(cfg.STEP6_INPUT_DIR).glob("*"):
    if old_path.is_file():
        old_path.unlink()
    elif old_path.is_dir():
        shutil.rmtree(old_path)

# Clear old Step 6 outputs.
if os.path.exists(cfg.STEP6_OUTPUT_DIR):
    shutil.rmtree(cfg.STEP6_OUTPUT_DIR)

os.makedirs(cfg.STEP6_OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 5. Use Step 5 zip from Google Drive
# ------------------------------------------------------------

step5_zip_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expA_settingA_6fps/settingA/data_expA_6fps_settingA/p4_settingA_step5_hidden_states_Qwen2_5_3B.zip"
)

print("\nUsing Step 5 zip from Google Drive:")
print(step5_zip_path)

if not step5_zip_path.exists():
    raise FileNotFoundError(f"Step 5 zip not found: {step5_zip_path}")

# ------------------------------------------------------------
# 6. Unzip Step 5 .pt files into STEP6_INPUT_DIR
# ------------------------------------------------------------

with zipfile.ZipFile(str(step5_zip_path), "r") as zf:
    zf.extractall(cfg.STEP6_INPUT_DIR)

pt_files = sorted(Path(cfg.STEP6_INPUT_DIR).rglob("*.pt"))

if not pt_files:
    raise FileNotFoundError(
        f"No .pt files found after extracting Step 5 zip into {cfg.STEP6_INPUT_DIR}"
    )

print("\nSetup complete.")
print("Config copied to:", config_target_path)
print("Step 5 zip:", step5_zip_path)
print("Number of .pt files found:", len(pt_files))
print("STEP6_INPUT_DIR:", cfg.STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", cfg.STEP6_OUTPUT_DIR)

for p in pt_files[:10]:
    print("-", p)

Mounted at /content/drive
Upload config.py


Saving config_exp_A_positional_context_6floorplans.py to config_exp_A_positional_context_6floorplans.py

Loaded config:
PILOT_ROOT: /content/pilot_4
DATA_DIR: /content/pilot_4/data
STEP6_INPUT_DIR: /content/pilot_4/data/step5_hidden_states_input
STEP6_OUTPUT_DIR: /content/pilot_4/data/step6_scene_split_single_direction_train_preserve_text_direction_outputs
STEP6_EXPERIMENT_NAME: scene_split_single_direction_train_preserve_text_direction

Using Step 5 zip from Google Drive:
/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expA_settingA_6fps/settingA/data_expA_6fps_settingA/p4_settingA_step5_hidden_states_Qwen2_5_3B.zip

Setup complete.
Config copied to: /content/pilot_4/config.py
Step 5 zip: /content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expA_settingA_6fps/settingA/data_expA_6fps_settingA/p4_settingA_step5_hidden_states_Qwen2_5_3B.zip
Number of .pt files found: 6
STEP6_INPUT_DIR: /content/pilot_4/data/step5_hidden_states_input
STEP6_OUTPUT_DIR: /content/pilot_4/dat

In [8]:
# ============================================================
# Imports and Step 6 configuration
# ============================================================

import os
import sys
import json
import random
import importlib
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_4.config as cfg
importlib.reload(cfg)

# ------------------------------------------------------------
# Basic config
# ------------------------------------------------------------

EXPERIMENT_NAME = cfg.STEP6_EXPERIMENT_NAME
RANDOM_SEED = getattr(cfg, "RANDOM_SEED", 42)

MODEL_NAME = getattr(cfg, "STEP5_MODEL_NAME", "Qwen/Qwen2.5-3B")
MODEL_TAG = getattr(cfg, "STEP5_MODEL_TAG", "Qwen2_5_3B")

STEP6_INPUT_DIR = cfg.STEP6_INPUT_DIR
STEP6_OUTPUT_DIR = cfg.STEP6_OUTPUT_DIR

FEATURE_KEY = cfg.STEP6_FEATURE_KEY
LABEL_FIELD = cfg.STEP6_LABEL_FIELD

TRAIN_SCENES = list(cfg.STEP6_TRAIN_SCENES)
TEST_SCENES = list(cfg.STEP6_TEST_SCENES)
REQUIRE_EXPLICIT_SCENE_SPLIT = getattr(cfg, "STEP6_REQUIRE_EXPLICIT_SCENE_SPLIT", True)

LOGREG_MAX_ITER = getattr(cfg, "STEP6_LOGREG_MAX_ITER", 5000)
LOGREG_C = getattr(cfg, "STEP6_LOGREG_C", 1.0)

EXPLICIT_DIRECT = cfg.EXPLICIT_DIRECT
EXPLICIT_INVERSE = cfg.EXPLICIT_INVERSE

PRINT_DATASET_SUMMARY = getattr(cfg, "STEP6_PRINT_DATASET_SUMMARY", True)
PRINT_LAYER_PROGRESS = getattr(cfg, "STEP6_PRINT_LAYER_PROGRESS", True)
PRINT_TOP_LAYERS = getattr(cfg, "STEP6_PRINT_TOP_LAYERS", True)
NUM_TOP_LAYERS_TO_PRINT = getattr(cfg, "STEP6_NUM_TOP_LAYERS_TO_PRINT", 10)

# ------------------------------------------------------------
# New single-direction training filter config
# ------------------------------------------------------------

TRAIN_ONE_DIRECTION_PER_PAIR_GROUP = getattr(
    cfg,
    "STEP6_TRAIN_ONE_DIRECTION_PER_PAIR_GROUP",
    False,
)

TRAIN_DIRECTION_SELECTION_MODE = getattr(
    cfg,
    "STEP6_TRAIN_DIRECTION_SELECTION_MODE",
    "random",
)

APPLY_DIRECTION_FILTER_TO_TRAIN_ONLY = getattr(
    cfg,
    "STEP6_APPLY_DIRECTION_FILTER_TO_TRAIN_ONLY",
    True,
)

DIRECTION_FILTER_GROUP_KEY = getattr(
    cfg,
    "STEP6_DIRECTION_FILTER_GROUP_KEY",
    "pair_group_id",
)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.makedirs(STEP6_OUTPUT_DIR, exist_ok=True)

print("Step 6 config loaded.")
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_TAG:", MODEL_TAG)
print("STEP6_INPUT_DIR:", STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", STEP6_OUTPUT_DIR)
print("FEATURE_KEY:", FEATURE_KEY)
print("LABEL_FIELD:", LABEL_FIELD)
print("TRAIN_SCENES:", TRAIN_SCENES)
print("TEST_SCENES:", TEST_SCENES)
print("EXPLICIT_DIRECT:", EXPLICIT_DIRECT)
print("EXPLICIT_INVERSE:", EXPLICIT_INVERSE)

print("\nTraining direction filter:")
print("TRAIN_ONE_DIRECTION_PER_PAIR_GROUP:", TRAIN_ONE_DIRECTION_PER_PAIR_GROUP)
print("TRAIN_DIRECTION_SELECTION_MODE:", TRAIN_DIRECTION_SELECTION_MODE)
print("APPLY_DIRECTION_FILTER_TO_TRAIN_ONLY:", APPLY_DIRECTION_FILTER_TO_TRAIN_ONLY)
print("DIRECTION_FILTER_GROUP_KEY:", DIRECTION_FILTER_GROUP_KEY)

Step 6 config loaded.
EXPERIMENT_NAME: scene_split_single_direction_train_preserve_text_direction
MODEL_NAME: Qwen/Qwen2.5-3B
MODEL_TAG: Qwen2_5_3B
STEP6_INPUT_DIR: /content/pilot_4/data/step5_hidden_states_input
STEP6_OUTPUT_DIR: /content/pilot_4/data/step6_scene_split_single_direction_train_preserve_text_direction_outputs
FEATURE_KEY: layer_diff_features
LABEL_FIELD: relation
TRAIN_SCENES: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4']
TEST_SCENES: ['FloorPlan5', 'FloorPlan6']
EXPLICIT_DIRECT: explicit_direct
EXPLICIT_INVERSE: explicit_inverse_or_same_sentence_pair

Training direction filter:
TRAIN_ONE_DIRECTION_PER_PAIR_GROUP: True
TRAIN_DIRECTION_SELECTION_MODE: random
APPLY_DIRECTION_FILTER_TO_TRAIN_ONLY: True
DIRECTION_FILTER_GROUP_KEY: pair_group_id


In [9]:
# ============================================================
# Helper functions for loading Step 5 records
# ============================================================

def discover_step5_pt_files(input_dir: str) -> List[str]:
    pt_paths = sorted(Path(input_dir).rglob("*.pt"))

    if not pt_paths:
        raise FileNotFoundError(f"No .pt files found in STEP6_INPUT_DIR: {input_dir}")

    return [str(p) for p in pt_paths]


def normalize_record(raw_record: Dict[str, Any], source_pt_file: str) -> Dict[str, Any]:
    """
    Normalize one Step 5 record into a predictable structure.

    Expected fields:
      - FEATURE_KEY, e.g. layer_diff_features
      - LABEL_FIELD, e.g. relation
      - metadata fields such as scene, evidence_type, pair_group_id
    """

    rec = dict(raw_record)
    rec["_source_pt_file"] = source_pt_file

    # Some versions save metadata as rec["metadata"], while others flatten it.
    metadata = rec.get("metadata", {})

    if metadata is None:
        metadata = {}

    if not isinstance(metadata, dict):
        metadata = {}

    # Flatten useful metadata into rec if absent.
    for k, v in metadata.items():
        rec.setdefault(k, v)

    # If Step 5 saved nested evidence/probe_pair, expose useful fields.
    evidence = rec.get("evidence", {})
    if isinstance(evidence, dict):
        rec.setdefault("evidence_type", evidence.get("evidence_type"))
        rec.setdefault("probe_direction_relative_to_text", evidence.get("probe_direction_relative_to_text"))
        rec.setdefault("text_relation_label", evidence.get("text_relation_label"))
        rec.setdefault("evidence_sentence", evidence.get("evidence_sentence"))

    probe_pair = rec.get("probe_pair", {})
    if isinstance(probe_pair, dict):
        rec.setdefault("probe_subject_uid", probe_pair.get("probe_subject_uid"))
        rec.setdefault("probe_object_uid", probe_pair.get("probe_object_uid"))
        rec.setdefault("probe_relation_label", probe_pair.get("probe_relation_label"))
        rec.setdefault("is_inverse_of_text_relation", probe_pair.get("is_inverse_of_text_relation"))

    # Standardize relation label.
    if LABEL_FIELD not in rec:
        if "relation" in rec:
            rec[LABEL_FIELD] = rec["relation"]
        elif "probe_relation_label" in rec:
            rec[LABEL_FIELD] = rec["probe_relation_label"]
        elif isinstance(probe_pair, dict) and "probe_relation_label" in probe_pair:
            rec[LABEL_FIELD] = probe_pair["probe_relation_label"]
        else:
            raise KeyError(f"Could not find label field '{LABEL_FIELD}' in record.")

    # Standardize scene.
    if "scene" not in rec:
        if "source_step3_scene" in rec:
            rec["scene"] = rec["source_step3_scene"]
        else:
            raise KeyError("Could not find scene field in record.")

    return rec


def extract_records_from_payload(payload: Any, source_pt_file: str) -> List[Dict[str, Any]]:
    """
    Supports several possible Step 5 payload formats:
      - list[record]
      - dict with key "records"
      - dict with key "data"
    """

    if isinstance(payload, list):
        raw_records = payload
    elif isinstance(payload, dict):
        if "records" in payload:
            raw_records = payload["records"]
        elif "data" in payload:
            raw_records = payload["data"]
        elif "examples" in payload:
            raw_records = payload["examples"]
        else:
            raise KeyError(
                f"Unsupported .pt payload keys in {source_pt_file}: {list(payload.keys())}"
            )
    else:
        raise TypeError(f"Unsupported .pt payload type in {source_pt_file}: {type(payload)}")

    records = [
        normalize_record(r, source_pt_file=source_pt_file)
        for r in raw_records
    ]

    return records


def to_numpy_feature(x: Any) -> np.ndarray:
    if isinstance(x, torch.Tensor):
        # NumPy cannot directly convert torch.bfloat16 tensors.
        # Move to CPU and convert to float32 before calling numpy().
        return x.detach().cpu().float().numpy()

    arr = np.asarray(x)

    if arr.dtype == np.dtype("O"):
        arr = arr.astype(np.float32)

    return arr.astype(np.float32, copy=False)

In [10]:
# ============================================================
# Load Step 5 records and build feature tensor
# ============================================================

pt_files = discover_step5_pt_files(STEP6_INPUT_DIR)

all_records = []

for pt_path in pt_files:
    payload = torch.load(pt_path, map_location="cpu")
    records = extract_records_from_payload(payload, source_pt_file=pt_path)
    all_records.extend(records)

if not all_records:
    raise ValueError("No Step 5 records loaded.")

print("Number of Step 5 records loaded:", len(all_records))
print("Number of source .pt files:", len(pt_files))

# ------------------------------------------------------------
# Validate feature and label fields
# ------------------------------------------------------------

missing_feature = [i for i, r in enumerate(all_records) if FEATURE_KEY not in r]
missing_label = [i for i, r in enumerate(all_records) if LABEL_FIELD not in r]

if missing_feature:
    raise KeyError(f"{len(missing_feature)} records are missing feature key: {FEATURE_KEY}")

if missing_label:
    raise KeyError(f"{len(missing_label)} records are missing label field: {LABEL_FIELD}")

# ------------------------------------------------------------
# Stack features
# Expected shape per record: [num_layers, hidden_dim]
# ------------------------------------------------------------

features = []

for r in all_records:
    f = to_numpy_feature(r[FEATURE_KEY])

    if f.ndim != 2:
        raise ValueError(
            f"Expected feature shape [num_layers, dim], got {f.shape} "
            f"for sample_id={r.get('sample_id')}"
        )

    features.append(f)

X_all = np.stack(features, axis=0)  # [num_samples, num_layers, dim]
y_all = np.array([r[LABEL_FIELD] for r in all_records])
metadata = all_records

num_samples, num_layers, feature_dim = X_all.shape

print("X_all shape:", X_all.shape)
print("num_samples:", num_samples)
print("num_layers:", num_layers)
print("feature_dim:", feature_dim)
print("Label counts:")
print(dict(Counter(y_all)))

print("\nScene counts:")
print(dict(Counter(r["scene"] for r in metadata)))

print("\nEvidence type counts:")
print(dict(Counter(r.get("evidence_type", "unknown") for r in metadata)))

Number of Step 5 records loaded: 3782
Number of source .pt files: 6
X_all shape: (3782, 37, 2048)
num_samples: 3782
num_layers: 37
feature_dim: 2048
Label counts:
{np.str_('above'): 330, np.str_('right_of'): 951, np.str_('contains'): 48, np.str_('on'): 435, np.str_('supports'): 435, np.str_('below'): 330, np.str_('in'): 48, np.str_('left_of'): 951, np.str_('near'): 254}

Scene counts:
{'FloorPlan1': 708, 'FloorPlan2': 582, 'FloorPlan3': 564, 'FloorPlan4': 580, 'FloorPlan5': 708, 'FloorPlan6': 640}

Evidence type counts:
{'explicit_inverse_or_same_sentence_pair': 1872, 'explicit_direct': 1910}


In [11]:
# ============================================================
# Scene split and initial common-label filtering
# ============================================================

all_scenes = sorted(set(r["scene"] for r in metadata))

if REQUIRE_EXPLICIT_SCENE_SPLIT:
    missing_train = sorted(set(TRAIN_SCENES) - set(all_scenes))
    missing_test = sorted(set(TEST_SCENES) - set(all_scenes))

    if missing_train or missing_test:
        raise ValueError(
            f"Scene split references missing scenes. "
            f"missing_train={missing_train}, missing_test={missing_test}, "
            f"available_scenes={all_scenes}"
        )

train_idx = np.array(
    [i for i, r in enumerate(metadata) if r["scene"] in TRAIN_SCENES],
    dtype=np.int64,
)

test_idx = np.array(
    [i for i, r in enumerate(metadata) if r["scene"] in TEST_SCENES],
    dtype=np.int64,
)

if len(train_idx) == 0:
    raise ValueError("No training examples selected.")

if len(test_idx) == 0:
    raise ValueError("No test examples selected.")

# Initial common-label filter before direction filtering.
train_labels_before = set(y_all[train_idx])
test_labels_before = set(y_all[test_idx])
common_labels_initial = sorted(train_labels_before & test_labels_before)

removed_train_only_labels_initial = sorted(train_labels_before - test_labels_before)
removed_test_only_labels_initial = sorted(test_labels_before - train_labels_before)

train_idx = np.array(
    [idx for idx in train_idx if y_all[idx] in common_labels_initial],
    dtype=np.int64,
)

test_idx = np.array(
    [idx for idx in test_idx if y_all[idx] in common_labels_initial],
    dtype=np.int64,
)

scene_split_info = {
    "train_scenes": TRAIN_SCENES,
    "test_scenes": TEST_SCENES,
    "num_train_before_common_label_filter": int(sum(r["scene"] in TRAIN_SCENES for r in metadata)),
    "num_test_before_common_label_filter": int(sum(r["scene"] in TEST_SCENES for r in metadata)),
    "train_labels_before": sorted(train_labels_before),
    "test_labels_before": sorted(test_labels_before),
    "common_labels_initial": common_labels_initial,
    "removed_train_only_labels_initial": removed_train_only_labels_initial,
    "removed_test_only_labels_initial": removed_test_only_labels_initial,
    "num_train_after_initial_common_label_filter": int(len(train_idx)),
    "num_test_after_initial_common_label_filter": int(len(test_idx)),
}

print("Initial scene split complete.")
print(json.dumps(scene_split_info, indent=2, ensure_ascii=False))

print("\nTrain label counts after initial common-label filter:")
print(dict(Counter(y_all[train_idx])))

print("\nTest label counts after initial common-label filter:")
print(dict(Counter(y_all[test_idx])))

print("\nTrain evidence type counts:")
print(dict(Counter(metadata[idx].get("evidence_type", "unknown") for idx in train_idx)))

print("\nTest evidence type counts:")
print(dict(Counter(metadata[idx].get("evidence_type", "unknown") for idx in test_idx)))

Initial scene split complete.
{
  "train_scenes": [
    "FloorPlan1",
    "FloorPlan2",
    "FloorPlan3",
    "FloorPlan4"
  ],
  "test_scenes": [
    "FloorPlan5",
    "FloorPlan6"
  ],
  "num_train_before_common_label_filter": 2434,
  "num_test_before_common_label_filter": 1348,
  "train_labels_before": [
    "above",
    "below",
    "contains",
    "in",
    "left_of",
    "near",
    "on",
    "right_of",
    "supports"
  ],
  "test_labels_before": [
    "above",
    "below",
    "contains",
    "in",
    "left_of",
    "near",
    "on",
    "right_of",
    "supports"
  ],
  "common_labels_initial": [
    "above",
    "below",
    "contains",
    "in",
    "left_of",
    "near",
    "on",
    "right_of",
    "supports"
  ],
  "removed_train_only_labels_initial": [],
  "removed_test_only_labels_initial": [],
  "num_train_after_initial_common_label_filter": 2434,
  "num_test_after_initial_common_label_filter": 1348
}

Train label counts after initial common-label filter:
{np.str_('a

In [12]:
# ============================================================
# Optional training filter:
# keep only one direction per direct/inverse pair group in TRAIN
# ============================================================

def get_direction_filter_group_key(meta: Dict[str, Any]) -> Any:
    key_name = DIRECTION_FILTER_GROUP_KEY

    if key_name in meta and meta[key_name] is not None:
        return meta[key_name]

    # Fallback if pair_group_id is missing.
    scene = meta.get("scene")
    paragraph_id = meta.get("paragraph_id")
    evidence_sentence = meta.get("evidence_sentence")

    s = (
        meta.get("probe_subject_uid")
        or meta.get("subject_alias")
        or meta.get("subject_uid")
    )

    o = (
        meta.get("probe_object_uid")
        or meta.get("object_alias")
        or meta.get("object_uid")
    )

    unordered_pair = tuple(sorted([str(s), str(o)]))

    return (scene, paragraph_id, evidence_sentence, unordered_pair)


def filter_train_one_direction_per_pair_group(
    train_idx: np.ndarray,
    metadata: List[Dict[str, Any]],
    seed: int,
    mode: str,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    rng = random.Random(seed)

    groups = defaultdict(list)

    for idx in train_idx:
        meta = metadata[int(idx)]
        group_key = get_direction_filter_group_key(meta)
        groups[group_key].append(int(idx))

    kept_indices = []

    summary = {
        "num_groups": 0,
        "num_groups_with_both_directions": 0,
        "num_groups_direct_only": 0,
        "num_groups_inverse_only": 0,
        "num_groups_other": 0,
        "kept_direct": 0,
        "kept_inverse": 0,
        "kept_other": 0,
        "selection_mode": mode,
        "random_seed": seed,
    }

    for group_key, idxs in groups.items():
        summary["num_groups"] += 1

        direct_idxs = [
            idx for idx in idxs
            if metadata[idx].get("evidence_type") == EXPLICIT_DIRECT
        ]

        inverse_idxs = [
            idx for idx in idxs
            if metadata[idx].get("evidence_type") == EXPLICIT_INVERSE
        ]

        if direct_idxs and inverse_idxs:
            summary["num_groups_with_both_directions"] += 1

            if mode == "direct":
                chosen = direct_idxs
            elif mode == "inverse":
                chosen = inverse_idxs
            elif mode == "random":
                chosen = direct_idxs if rng.random() < 0.5 else inverse_idxs
            else:
                raise ValueError(f"Unknown direction selection mode: {mode}")

        elif direct_idxs:
            summary["num_groups_direct_only"] += 1
            chosen = direct_idxs

        elif inverse_idxs:
            summary["num_groups_inverse_only"] += 1
            chosen = inverse_idxs

        else:
            summary["num_groups_other"] += 1
            chosen = idxs

        for idx in chosen:
            et = metadata[idx].get("evidence_type")

            if et == EXPLICIT_DIRECT:
                summary["kept_direct"] += 1
            elif et == EXPLICIT_INVERSE:
                summary["kept_inverse"] += 1
            else:
                summary["kept_other"] += 1

        kept_indices.extend(chosen)

    kept_indices = np.array(sorted(kept_indices), dtype=np.int64)

    return kept_indices, summary


train_direction_filter_info = {
    "enabled": bool(TRAIN_ONE_DIRECTION_PER_PAIR_GROUP),
    "mode": TRAIN_DIRECTION_SELECTION_MODE,
    "apply_to_train_only": APPLY_DIRECTION_FILTER_TO_TRAIN_ONLY,
    "group_key": DIRECTION_FILTER_GROUP_KEY,
    "num_train_before_direction_filter": int(len(train_idx)),
    "num_train_after_direction_filter": int(len(train_idx)),
    "summary": None,
}

if TRAIN_ONE_DIRECTION_PER_PAIR_GROUP:
    if not APPLY_DIRECTION_FILTER_TO_TRAIN_ONLY:
        raise NotImplementedError(
            "This notebook currently supports direction filtering for TRAIN only. "
            "Set STEP6_APPLY_DIRECTION_FILTER_TO_TRAIN_ONLY = True."
        )

    train_idx_before_direction_filter = train_idx.copy()

    train_idx, direction_filter_summary = filter_train_one_direction_per_pair_group(
        train_idx=train_idx,
        metadata=metadata,
        seed=RANDOM_SEED,
        mode=TRAIN_DIRECTION_SELECTION_MODE,
    )

    train_direction_filter_info["num_train_after_direction_filter"] = int(len(train_idx))
    train_direction_filter_info["summary"] = direction_filter_summary

    print("Applied one-direction-per-pair-group filter to TRAIN split.")
    print("Train examples before:", len(train_idx_before_direction_filter))
    print("Train examples after:", len(train_idx))
    print(json.dumps(train_direction_filter_info, indent=2, ensure_ascii=False))
else:
    print("Training direction filter disabled.")

Applied one-direction-per-pair-group filter to TRAIN split.
Train examples before: 2434
Train examples after: 1232
{
  "enabled": true,
  "mode": "random",
  "apply_to_train_only": true,
  "group_key": "pair_group_id",
  "num_train_before_direction_filter": 2434,
  "num_train_after_direction_filter": 1232,
  "summary": {
    "num_groups": 1232,
    "num_groups_with_both_directions": 1202,
    "num_groups_direct_only": 30,
    "num_groups_inverse_only": 0,
    "num_groups_other": 0,
    "kept_direct": 615,
    "kept_inverse": 617,
    "kept_other": 0,
    "selection_mode": "random",
    "random_seed": 42
  }
}


In [13]:
# ============================================================
# Final label consistency after training-direction filter
#
# Direction filtering may remove rare labels from train.
# Recompute common labels and filter test accordingly.
# ============================================================

train_labels_after_direction = set(y_all[train_idx])
test_labels_after_direction = set(y_all[test_idx])

common_labels_final = sorted(train_labels_after_direction & test_labels_after_direction)

removed_train_only_labels_final = sorted(train_labels_after_direction - test_labels_after_direction)
removed_test_only_labels_final = sorted(test_labels_after_direction - train_labels_after_direction)

train_idx = np.array(
    [idx for idx in train_idx if y_all[idx] in common_labels_final],
    dtype=np.int64,
)

test_idx = np.array(
    [idx for idx in test_idx if y_all[idx] in common_labels_final],
    dtype=np.int64,
)

label_order = common_labels_final

scene_split_info.update({
    "train_direction_filter_info": train_direction_filter_info,
    "train_labels_after_direction_filter": sorted(train_labels_after_direction),
    "test_labels_after_direction_filter": sorted(test_labels_after_direction),
    "common_labels_final": common_labels_final,
    "removed_train_only_labels_final": removed_train_only_labels_final,
    "removed_test_only_labels_final": removed_test_only_labels_final,
    "num_train_final": int(len(train_idx)),
    "num_test_final": int(len(test_idx)),
    "train_label_counts_final": dict(Counter(y_all[train_idx])),
    "test_label_counts_final": dict(Counter(y_all[test_idx])),
    "train_evidence_type_counts_final": dict(
        Counter(metadata[idx].get("evidence_type", "unknown") for idx in train_idx)
    ),
    "test_evidence_type_counts_final": dict(
        Counter(metadata[idx].get("evidence_type", "unknown") for idx in test_idx)
    ),
})

print("Final train/test indices prepared.")
print(json.dumps(scene_split_info, indent=2, ensure_ascii=False))

print("\nFinal label_order:")
print(label_order)

print("\nFinal train label counts:")
print(dict(Counter(y_all[train_idx])))

print("\nFinal test label counts:")
print(dict(Counter(y_all[test_idx])))

print("\nFinal train evidence type counts:")
print(dict(Counter(metadata[idx].get("evidence_type", "unknown") for idx in train_idx)))

print("\nFinal test evidence type counts:")
print(dict(Counter(metadata[idx].get("evidence_type", "unknown") for idx in test_idx)))

Final train/test indices prepared.
{
  "train_scenes": [
    "FloorPlan1",
    "FloorPlan2",
    "FloorPlan3",
    "FloorPlan4"
  ],
  "test_scenes": [
    "FloorPlan5",
    "FloorPlan6"
  ],
  "num_train_before_common_label_filter": 2434,
  "num_test_before_common_label_filter": 1348,
  "train_labels_before": [
    "above",
    "below",
    "contains",
    "in",
    "left_of",
    "near",
    "on",
    "right_of",
    "supports"
  ],
  "test_labels_before": [
    "above",
    "below",
    "contains",
    "in",
    "left_of",
    "near",
    "on",
    "right_of",
    "supports"
  ],
  "common_labels_initial": [
    "above",
    "below",
    "contains",
    "in",
    "left_of",
    "near",
    "on",
    "right_of",
    "supports"
  ],
  "removed_train_only_labels_initial": [],
  "removed_test_only_labels_initial": [],
  "num_train_after_initial_common_label_filter": 2434,
  "num_test_after_initial_common_label_filter": 1348,
  "train_direction_filter_info": {
    "enabled": true,
    "m

In [14]:
# ============================================================
# Build test subsets
# ============================================================

test_overall_idx = test_idx

test_direct_idx = np.array(
    [
        idx for idx in test_idx
        if metadata[int(idx)].get("evidence_type") == EXPLICIT_DIRECT
    ],
    dtype=np.int64,
)

test_inverse_idx = np.array(
    [
        idx for idx in test_idx
        if metadata[int(idx)].get("evidence_type") == EXPLICIT_INVERSE
    ],
    dtype=np.int64,
)

test_subset_indices = {
    "test_overall": test_overall_idx,
    "test_direct": test_direct_idx,
    "test_inverse": test_inverse_idx,
}

test_subset_summary = {}

for subset_key, idxs in test_subset_indices.items():
    test_subset_summary[subset_key] = {
        "num_examples": int(len(idxs)),
        "label_counts": dict(Counter(y_all[idxs])) if len(idxs) else {},
        "evidence_type_counts": dict(
            Counter(metadata[int(idx)].get("evidence_type", "unknown") for idx in idxs)
        ) if len(idxs) else {},
    }

print("Test subset summary:")
print(json.dumps(test_subset_summary, indent=2, ensure_ascii=False))

Test subset summary:
{
  "test_overall": {
    "num_examples": 1348,
    "label_counts": {
      "right_of": 329,
      "above": 132,
      "below": 132,
      "left_of": 329,
      "contains": 8,
      "in": 8,
      "near": 100,
      "supports": 155,
      "on": 155
    },
    "evidence_type_counts": {
      "explicit_inverse_or_same_sentence_pair": 670,
      "explicit_direct": 678
    }
  },
  "test_direct": {
    "num_examples": 678,
    "label_counts": {
      "above": 73,
      "below": 59,
      "left_of": 161,
      "contains": 5,
      "right_of": 168,
      "near": 54,
      "on": 96,
      "supports": 59,
      "in": 3
    },
    "evidence_type_counts": {
      "explicit_direct": 678
    }
  },
  "test_inverse": {
    "num_examples": 670,
    "label_counts": {
      "right_of": 161,
      "below": 73,
      "above": 59,
      "in": 5,
      "left_of": 168,
      "near": 46,
      "supports": 96,
      "on": 59,
      "contains": 3
    },
    "evidence_type_counts": {
     

In [15]:
# ============================================================
# Layer-wise training and evaluation
# ============================================================

def evaluate_classifier(
    clf: LogisticRegression,
    X_eval: np.ndarray,
    y_eval: np.ndarray,
    labels: List[str],
) -> Dict[str, Any]:
    if len(y_eval) == 0:
        return {
            "num_examples": 0,
            "accuracy": None,
            "macro_f1": None,
            "label_order": labels,
            "classification_report": {},
            "confusion_matrix": [],
        }

    y_pred = clf.predict(X_eval)

    acc = accuracy_score(y_eval, y_pred)
    macro_f1 = f1_score(y_eval, y_pred, labels=labels, average="macro", zero_division=0)

    report = classification_report(
        y_eval,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_eval,
        y_pred,
        labels=labels,
    )

    return {
        "num_examples": int(len(y_eval)),
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "label_order": labels,
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
    }


results_by_layer = []

for layer_id in range(num_layers):
    if PRINT_LAYER_PROGRESS:
        print(f"Training layer {layer_id}/{num_layers - 1}")

    X_train_layer = X_all[train_idx, layer_id, :]
    y_train_layer = y_all[train_idx]

    clf = LogisticRegression(
        max_iter=LOGREG_MAX_ITER,
        C=LOGREG_C,
        multi_class="auto",
        solver="lbfgs",
        random_state=RANDOM_SEED,
    )

    clf.fit(X_train_layer, y_train_layer)

    layer_result = {
        "layer": int(layer_id),
        "train": evaluate_classifier(
            clf,
            X_train_layer,
            y_train_layer,
            labels=label_order,
        ),
    }

    for subset_key, idxs in test_subset_indices.items():
        X_eval = X_all[idxs, layer_id, :]
        y_eval = y_all[idxs]

        layer_result[subset_key] = evaluate_classifier(
            clf,
            X_eval,
            y_eval,
            labels=label_order,
        )

    results_by_layer.append(layer_result)

print("Layer-wise training and evaluation complete.")

Training layer 0/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training layer 1/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training layer 2/36


...

Training layer 35/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training layer 36/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer-wise training and evaluation complete.


In [16]:
# ============================================================
# Summarize top layers
# ============================================================

summary_rows = []

for r in results_by_layer:
    row = {
        "layer": r["layer"],
        "train_accuracy": r["train"]["accuracy"],
        "train_macro_f1": r["train"]["macro_f1"],
    }

    for subset_key in ["test_overall", "test_direct", "test_inverse"]:
        row[f"{subset_key}_accuracy"] = r[subset_key]["accuracy"]
        row[f"{subset_key}_macro_f1"] = r[subset_key]["macro_f1"]
        row[f"{subset_key}_num_examples"] = r[subset_key]["num_examples"]

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_csv_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"step6_{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}_layer_summary.csv",
)

summary_df.to_csv(summary_csv_path, index=False)

print("Saved layer summary CSV:")
print(summary_csv_path)

display(summary_df.head())

if PRINT_TOP_LAYERS:
    for metric in [
        "test_overall_accuracy",
        "test_overall_macro_f1",
        "test_direct_accuracy",
        "test_inverse_accuracy",
    ]:
        print("\n" + "=" * 80)
        print("Top layers by:", metric)
        print("=" * 80)

        cols = [
            "layer",
            metric,
            "test_overall_accuracy",
            "test_overall_macro_f1",
            "test_direct_accuracy",
            "test_inverse_accuracy",
        ]

        display(
            summary_df.sort_values(metric, ascending=False)
            .loc[:, cols]
            .head(NUM_TOP_LAYERS_TO_PRINT)
        )

Saved layer summary CSV:
/content/pilot_4/data/step6_scene_split_single_direction_train_preserve_text_direction_outputs/step6_scene_split_single_direction_train_preserve_text_direction_Qwen2_5_3B_layer_diff_features_layer_summary.csv


,layer,train_accuracy,train_macro_f1,test_overall_accuracy,test_overall_macro_f1,test_overall_num_examples,test_direct_accuracy,test_direct_macro_f1,test_direct_num_examples,test_inverse_accuracy,test_inverse_macro_f1,test_inverse_num_examples
0,0,0.646916,0.428400,0.435460,0.288677,1348,0.438053,0.298509,678,0.432836,0.281454,670
1,1,0.969968,0.964891,0.715875,0.597550,1348,0.735988,0.653871,678,0.695522,0.518846,670
2,2,0.991883,0.991911,0.795994,0.663788,1348,0.820059,0.717024,678,0.771642,0.582321,670
3,3,0.996753,0.996826,0.801187,0.671857,1348,0.823009,0.729171,678,0.779104,0.587657,670
4,4,0.998377,0.998430,0.760386,0.620849,1348,0.774336,0.692700,678,0.746269,0.533820,670



Top layers by: test_overall_accuracy


,layer,test_overall_accuracy,test_overall_accuracy,test_overall_macro_f1,test_direct_accuracy,test_inverse_accuracy
3,3,0.801187,0.801187,0.671857,0.823009,0.779104
2,2,0.795994,0.795994,0.663788,0.820059,0.771642
5,5,0.774481,0.774481,0.647087,0.781711,0.767164
17,17,0.770030,0.770030,0.680128,0.765487,0.774627
4,4,0.760386,0.760386,0.620849,0.774336,0.746269
19,19,0.743323,0.743323,0.646775,0.744838,0.741791
1,1,0.715875,0.715875,0.597550,0.735988,0.695522
20,20,0.709941,0.709941,0.623451,0.700590,0.719403
6,6,0.703264,0.703264,0.557833,0.699115,0.707463
18,18,0.701039,0.701039,0.612643,0.700590,0.701493



Top layers by: test_overall_macro_f1


,layer,test_overall_macro_f1,test_overall_accuracy,test_overall_macro_f1,test_direct_accuracy,test_inverse_accuracy
17,17,0.680128,0.770030,0.680128,0.765487,0.774627
3,3,0.671857,0.801187,0.671857,0.823009,0.779104
2,2,0.663788,0.795994,0.663788,0.820059,0.771642
5,5,0.647087,0.774481,0.647087,0.781711,0.767164
19,19,0.646775,0.743323,0.646775,0.744838,0.741791
20,20,0.623451,0.709941,0.623451,0.700590,0.719403
4,4,0.620849,0.760386,0.620849,0.774336,0.746269
18,18,0.612643,0.701039,0.612643,0.700590,0.701493
1,1,0.597550,0.715875,0.597550,0.735988,0.695522
10,10,0.575930,0.655786,0.575930,0.666667,0.644776



Top layers by: test_direct_accuracy


,layer,test_direct_accuracy,test_overall_accuracy,test_overall_macro_f1,test_direct_accuracy,test_inverse_accuracy
3,3,0.823009,0.801187,0.671857,0.823009,0.779104
2,2,0.820059,0.795994,0.663788,0.820059,0.771642
5,5,0.781711,0.774481,0.647087,0.781711,0.767164
4,4,0.774336,0.760386,0.620849,0.774336,0.746269
17,17,0.765487,0.770030,0.680128,0.765487,0.774627
19,19,0.744838,0.743323,0.646775,0.744838,0.741791
1,1,0.735988,0.715875,0.597550,0.735988,0.695522
18,18,0.700590,0.701039,0.612643,0.700590,0.701493
20,20,0.700590,0.709941,0.623451,0.700590,0.719403
6,6,0.699115,0.703264,0.557833,0.699115,0.707463



Top layers by: test_inverse_accuracy


,layer,test_inverse_accuracy,test_overall_accuracy,test_overall_macro_f1,test_direct_accuracy,test_inverse_accuracy
3,3,0.779104,0.801187,0.671857,0.823009,0.779104
17,17,0.774627,0.770030,0.680128,0.765487,0.774627
2,2,0.771642,0.795994,0.663788,0.820059,0.771642
5,5,0.767164,0.774481,0.647087,0.781711,0.767164
4,4,0.746269,0.760386,0.620849,0.774336,0.746269
19,19,0.741791,0.743323,0.646775,0.744838,0.741791
20,20,0.719403,0.709941,0.623451,0.700590,0.719403
6,6,0.707463,0.703264,0.557833,0.699115,0.707463
18,18,0.701493,0.701039,0.612643,0.700590,0.701493
1,1,0.695522,0.715875,0.597550,0.735988,0.695522


In [17]:
# ============================================================
# Save full Step 6 result JSON
# ============================================================

output_payload = {
    "experiment_name": EXPERIMENT_NAME,
    "description": (
        "Scene-split single-direction-train evaluation. "
        "Training uses one randomly selected direction per direct/inverse pair group "
        "from train scenes. Testing keeps all direct and inverse samples from test scenes, "
        "with separate evaluation for overall, direct subset, and inverse subset."
    ),
    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,
    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,
    "num_samples_loaded": int(num_samples),
    "num_layers": int(num_layers),
    "feature_dim": int(feature_dim),
    "label_order": label_order,
    "scene_split_info": scene_split_info,
    "test_subset_summary": test_subset_summary,
    "step6_config": {
        "experiment_name": EXPERIMENT_NAME,
        "model_name": MODEL_NAME,
        "model_tag": MODEL_TAG,
        "feature_key": FEATURE_KEY,
        "label_field": LABEL_FIELD,
        "train_scenes": TRAIN_SCENES,
        "test_scenes": TEST_SCENES,
        "logreg_max_iter": LOGREG_MAX_ITER,
        "logreg_c": LOGREG_C,
        "train_one_direction_per_pair_group": TRAIN_ONE_DIRECTION_PER_PAIR_GROUP,
        "train_direction_selection_mode": TRAIN_DIRECTION_SELECTION_MODE,
        "apply_direction_filter_to_train_only": APPLY_DIRECTION_FILTER_TO_TRAIN_ONLY,
        "direction_filter_group_key": DIRECTION_FILTER_GROUP_KEY,
        "random_seed": RANDOM_SEED,
    },
    "source_pt_files": pt_files,
    "results_by_layer": results_by_layer,
}

output_json_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"step6_{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}.json",
)

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, indent=2, ensure_ascii=False)

print("Saved full Step 6 result JSON:")
print(output_json_path)

Saved full Step 6 result JSON:
/content/pilot_4/data/step6_scene_split_single_direction_train_preserve_text_direction_outputs/step6_scene_split_single_direction_train_preserve_text_direction_Qwen2_5_3B_layer_diff_features.json


In [18]:
# ============================================================
# Zip Step 6 outputs
# ============================================================

zip_base = f"/content/pilot4_step6_{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}"
zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=STEP6_OUTPUT_DIR,
)

print("Created Step 6 output zip:")
print(zip_path)

Created Step 6 output zip:
/content/pilot4_step6_scene_split_single_direction_train_preserve_text_direction_Qwen2_5_3B_layer_diff_features.zip


In [19]:
# ============================================================
# Optional download
# ============================================================

from google.colab import files

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>